# 🔬 Feature Engineering
## Can new derived features improve basin generalisation?

**R&D Notebook** — Audit existing features, derive new ones, analyse predictive power, and measure impact on LOBO performance.

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
QUICK_MODE = True   # Set False for full-fidelity
TARGET_BASIN = 'SI'  # Holdout basin for experiments
EPOCHS = 3 if QUICK_MODE else 15
USE_3D = not QUICK_MODE
USE_ENV = True
BATCH_SIZE = 64 if QUICK_MODE else 32

In [ ]:
import os, glob, math, time, warnings, copy
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from netCDF4 import Dataset as NC4Dataset
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.metrics import f1_score, precision_recall_fscore_support
from scipy.stats import pointbiserialr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
matplotlib.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('/Users/thiruanand/MLfTCC')
DATA_ROOT    = PROJECT_ROOT / 'Notebooks' / 'data' / 'tropicyclonenet' / 'TCND_test'
BASINS = ['EP', 'NA', 'NI', 'SI', 'SP', 'WP']
BASIN_FULL = {'EP':'Eastern Pacific','NA':'North Atlantic','NI':'North Indian',
              'SI':'South Indian','SP':'South Pacific','WP':'Western Pacific'}
INT_NAMES = ['Weakening','Slow Weakening','Steady','Intensification']  # 4 classes from dataset (0-3)
RI_CLASS = 3  # Intensification is the highest class (class 3)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)

print(f"✓ Setup | Device: {DEVICE} | Quick: {QUICK_MODE}")

## 1. Data Loading

In [ ]:
def scan_files(data_root, basins, subdir, ext):
    idx = {}
    for b in basins:
        d = data_root / subdir / b
        if not d.exists(): continue
        for f in sorted(d.rglob(f'*{ext}')):
            storm = f.parent.name if ext=='.npy' else f.stem.split('_')[1]
            dt = f.stem if ext=='.npy' else f.stem.split('_')[2]
            idx[(b, storm, dt)] = str(f)
    return idx

def build_env_vector(env_dict):
    def to_scalar(x):
        return np.float32(np.argmax(x)) if isinstance(x, np.ndarray) else np.float32(max(x, 0))
    return np.concatenate([
        np.array(env_dict.get('month', np.zeros(12)), dtype=np.float32),
        np.array(env_dict.get('area', np.zeros(6)), dtype=np.float32),
        np.array(env_dict.get('intensity_class', np.zeros(6)), dtype=np.float32),
        np.array([env_dict.get('wind', 0)], dtype=np.float32),
        np.array([env_dict.get('move_velocity', 0)], dtype=np.float32),
        np.array(env_dict.get('location_long', np.zeros(36)), dtype=np.float32),
        np.array(env_dict.get('location_lat', np.zeros(12)), dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_direction12',-1))], dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_direction24',-1))], dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_inte_change24',-1))], dtype=np.float32),
    ])

def load_data3d(nc_path):
    with NC4Dataset(nc_path, 'r') as nc:
        u = np.array(nc.variables['u'][:], dtype=np.float32)
        v = np.array(nc.variables['v'][:], dtype=np.float32)
        z_geo = np.array(nc.variables['z'][:], dtype=np.float32)
        sst_var = nc.variables['sst']
        sst_raw = np.ma.filled(sst_var[:], fill_value=np.nan).astype(np.float32)
    chs = [u[0,i] for i in range(4)] + [v[0,i] for i in range(4)] + [z_geo[0,i] for i in range(4)]
    chs.append(sst_raw[0] if sst_raw.ndim==3 else sst_raw)
    data_3d = np.stack(chs, axis=0)
    for c in range(13):
        ch = data_3d[c]; valid = ch[~np.isnan(ch)]
        if len(valid) > 0:
            mu, std = valid.mean(), max(valid.std(), 1e-6)
            ch[np.isnan(ch)] = mu; data_3d[c] = (ch - mu) / std
        else: data_3d[c] = np.zeros_like(ch)
    return data_3d

def compute_labeled_dataset(data_root, basins):
    """Load all samples with CORRECT column order and targets from Env-Data."""
    records = []
    d3d_idx = scan_files(data_root, basins, 'Data3D', '.nc')
    env_idx = scan_files(data_root, basins, 'Env-Data', '.npy')
    # Correct column order (from src/data/dataset.py):
    # ID | FLAG | LAT_norm | LONG_norm | WND_norm | PRES_norm | YYYYMMDDHH | Name
    TCND_COLS = ['ID', 'FLAG', 'LAT_norm', 'LONG_norm', 'WND_norm', 'PRES_norm', 'YYYYMMDDHH', 'Name']
    for basin in basins:
        d = data_root / 'Data1D' / basin / 'test'
        if not d.exists(): continue
        for f in sorted(d.glob('*.txt')):
            try:
                df = pd.read_csv(f, sep='\t', header=None, names=TCND_COLS)
                stem = f.stem
                year = stem[len(basin):len(basin)+4]
                tc_name = stem[len(basin)+4+3:]  # skip 'BST'
                for _, r in df.iterrows():
                    dt = str(int(r['YYYYMMDDHH']))
                    storm = tc_name
                    d3d_p = d3d_idx.get((basin,storm.upper(),dt)) or d3d_idx.get((basin,storm,dt))
                    env_p = env_idx.get((basin,storm.upper(),dt)) or env_idx.get((basin,storm,dt))
                    if not (d3d_p and env_p): continue
                    # Read TARGETS from Env-Data (NOT computed from wind change)
                    env_d = np.load(env_p, allow_pickle=True).item()
                    y_int = int(np.asarray(env_d.get('future_inte_change24', 2)).ravel()[0])
                    y_dir = int(np.asarray(env_d.get('future_direction24', 0)).ravel()[0])
                    if y_int < 0: y_int = 2  # sentinel -1 → steady
                    if y_dir < 0: y_dir = 0
                    y_int = min(y_int, 4)
                    y_dir = min(y_dir, 7)
                    # data1d: [LONG, LAT, PRES, WND] (matching src/data/dataset.py)
                    records.append({'basin':basin,'storm':storm,'datetime':dt,
                        'data1d':np.array([r['LONG_norm'],r['LAT_norm'],r['PRES_norm'],r['WND_norm']],dtype=np.float32),
                        'data3d_path':d3d_p,'env_path':env_p,'y_intensity':y_int,'y_direction':y_dir})
            except: pass
    return records

print("Building labeled dataset...")
t0 = time.time()
ALL_SAMPLES = compute_labeled_dataset(DATA_ROOT, BASINS)
print(f"✓ {len(ALL_SAMPLES):,} samples in {time.time()-t0:.1f}s")

## 2. Existing Feature Audit
Catalogue what the model currently sees.

In [ ]:
feature_audit = pd.DataFrame([
    {'Group': 'Data1D', 'Feature': 'LONG_norm', 'Dim': 1, 'Description': 'Normalised longitude'},
    {'Group': 'Data1D', 'Feature': 'LAT_norm', 'Dim': 1, 'Description': 'Normalised latitude'},
    {'Group': 'Data1D', 'Feature': 'PRES_norm', 'Dim': 1, 'Description': 'Normalised pressure'},
    {'Group': 'Data1D', 'Feature': 'WND_norm', 'Dim': 1, 'Description': 'Normalised wind speed'},
    {'Group': 'Env-Data', 'Feature': 'month', 'Dim': 12, 'Description': 'One-hot month encoding'},
    {'Group': 'Env-Data', 'Feature': 'area', 'Dim': 6, 'Description': 'Basin area encoding'},
    {'Group': 'Env-Data', 'Feature': 'intensity_class', 'Dim': 6, 'Description': 'Intensity category'},
    {'Group': 'Env-Data', 'Feature': 'wind', 'Dim': 1, 'Description': 'Wind magnitude'},
    {'Group': 'Env-Data', 'Feature': 'move_velocity', 'Dim': 1, 'Description': 'Storm translation speed'},
    {'Group': 'Env-Data', 'Feature': 'location_long', 'Dim': 36, 'Description': 'Longitude bins'},
    {'Group': 'Env-Data', 'Feature': 'location_lat', 'Dim': 12, 'Description': 'Latitude bins'},
    {'Group': 'Env-Data', 'Feature': 'history_direction12', 'Dim': 1, 'Description': '12h motion direction'},
    {'Group': 'Env-Data', 'Feature': 'history_direction24', 'Dim': 1, 'Description': '24h motion direction'},
    {'Group': 'Env-Data', 'Feature': 'history_inte_change24', 'Dim': 1, 'Description': '24h intensity change'},
    {'Group': 'Physics', 'Feature': 'sst_anom (proxy)', 'Dim': 1, 'Description': 'SST anomaly from wind'},
    {'Group': 'Physics', 'Feature': 'wind_shear', 'Dim': 1, 'Description': '|pres - wind|'},
    {'Group': 'Physics', 'Feature': 'coriolis', 'Dim': 1, 'Description': 'Coriolis parameter'},
    {'Group': 'Physics', 'Feature': 'mpi_proxy', 'Dim': 1, 'Description': 'Max Potential Intensity proxy'},
    {'Group': 'Physics', 'Feature': 'bl_moisture', 'Dim': 1, 'Description': 'Boundary-layer moisture proxy'},
    {'Group': 'Physics', 'Feature': 'outflow_temp', 'Dim': 1, 'Description': 'Outflow temperature proxy'},
    {'Group': 'Physics', 'Feature': 'steering', 'Dim': 1, 'Description': 'Steering flow proxy'},
    {'Group': 'Physics', 'Feature': 'current_int', 'Dim': 1, 'Description': 'Current intensity'},
    {'Group': 'Data3D', 'Feature': 'u wind (4 levels)', 'Dim': '4×81×81', 'Description': 'Zonal wind at 4 pressure levels'},
    {'Group': 'Data3D', 'Feature': 'v wind (4 levels)', 'Dim': '4×81×81', 'Description': 'Meridional wind at 4 pressure levels'},
    {'Group': 'Data3D', 'Feature': 'geopotential (4 levels)', 'Dim': '4×81×81', 'Description': 'Geopotential height at 4 levels'},
    {'Group': 'Data3D', 'Feature': 'SST', 'Dim': '1×81×81', 'Description': 'Sea surface temperature'},
])

print("Existing Feature Audit:")
display(feature_audit)
print(f"\nTotal scalar dimensions: Data1D=4, Env=77, Physics=8, Data3D=13×81×81")

## 3. Candidate New Features
Derive features from existing data that may capture physics-relevant signals.

In [ ]:
def compute_enhanced_features(sample):
    """Compute new engineered features from a single sample."""
    lon, lat, pres, wnd = sample['data1d']
    
    # ── Wind-Pressure Relationship ──
    wind_pres_ratio = wnd / max(abs(pres), 1e-6)  # Deviation from wind-pressure relationship
    wind_pres_diff = wnd - pres  # Simple difference
    
    # ── Latitude-based features ──
    abs_lat = abs(lat)
    lat_rad = abs_lat * np.pi / 2  # Approximate radian conversion
    coriolis_enhanced = 2 * 7.2921e-5 * np.sin(np.clip(lat_rad, 0, np.pi/2))
    coriolis_x_wind = coriolis_enhanced * max(wnd, 0)  # Interaction
    
    # ── SST and thermodynamic proxies ──
    sst_proxy = wnd * 0.5  # Proxy from wind
    sst_bin = int(np.clip(sst_proxy * 5, 0, 4))  # Discretised SST anomaly
    
    # ── Shear-intensity interaction ──
    shear = abs(pres - wnd)
    shear_int_interaction = shear * wnd  # Shear × current intensity
    favourable_shear = float(shear < 0.15)  # Low shear indicator
    
    # ── Storm translation speed proxy ──
    translation_proxy = abs(lon) * 0.1  # Very rough
    
    # ── MPI-related ──
    mpi_deficit = max(0, sst_proxy * 2 - shear) - wnd  # Room to intensify
    intensification_potential = max(0, mpi_deficit)  # Clipped positive
    
    # ── Basin-invariant features (deliberately remove basin signal) ──
    wnd_squared = wnd ** 2  # Non-linear intensity
    pres_cubed = pres ** 3  # Non-linear pressure
    
    return {
        'wind_pres_ratio': float(wind_pres_ratio),
        'wind_pres_diff': float(wind_pres_diff),
        'abs_lat': float(abs_lat),
        'coriolis_enhanced': float(coriolis_enhanced),
        'coriolis_x_wind': float(coriolis_x_wind),
        'sst_proxy': float(sst_proxy),
        'sst_bin': float(sst_bin),
        'shear': float(shear),
        'shear_int_interaction': float(shear_int_interaction),
        'favourable_shear': float(favourable_shear),
        'translation_proxy': float(translation_proxy),
        'mpi_deficit': float(mpi_deficit),
        'intensification_potential': float(intensification_potential),
        'wnd_squared': float(wnd_squared),
        'pres_cubed': float(pres_cubed),
    }

# Compute for all samples
ENHANCED_FEATURE_NAMES = list(compute_enhanced_features(ALL_SAMPLES[0]).keys())
print(f"New engineered features ({len(ENHANCED_FEATURE_NAMES)}):")
for name in ENHANCED_FEATURE_NAMES:
    print(f"  • {name}")

## 4. Feature Distribution Analysis
Histogram per basin, box-plot per intensity class.

In [ ]:
# Build feature DataFrame
feat_rows = []
for s in ALL_SAMPLES:
    feats = compute_enhanced_features(s)
    feats['basin'] = s['basin']
    feats['y_intensity'] = s['y_intensity']
    feats['is_ri'] = int(s['y_intensity'] == RI_CLASS)
    feat_rows.append(feats)
feat_df = pd.DataFrame(feat_rows)

# Plot distributions of key features by basin
key_features = ['wind_pres_ratio', 'coriolis_x_wind', 'shear_int_interaction',
                'intensification_potential', 'favourable_shear', 'mpi_deficit']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.ravel(), key_features):
    for b in BASINS:
        data = feat_df[feat_df.basin==b][feat]
        ax.hist(data, bins=30, alpha=0.4, label=b, density=True)
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=7)
plt.suptitle('Engineered Feature Distributions by Basin', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Box-plots by intensity class
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.ravel(), key_features):
    feat_df.boxplot(column=feat, by='y_intensity', ax=ax)
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Intensity Class')
    ax.set_xticklabels([f'{i}\n{INT_NAMES[i][:6]}' for i in range(4)], fontsize=8)
plt.suptitle('Engineered Features by Intensity Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 5. Univariate Predictive Power
Mutual information, ANOVA F-test, and point-biserial correlation.

In [ ]:
X_feats = feat_df[ENHANCED_FEATURE_NAMES].values
y_int = feat_df['y_intensity'].values
y_ri = feat_df['is_ri'].values

# Mutual Information (vs intensity class)
mi_scores = mutual_info_classif(X_feats, y_int, random_state=SEED)

# ANOVA F-test (vs intensity class)
f_scores, f_pvals = f_classif(X_feats, y_int)

# Point-biserial correlation (vs RI binary)
pb_corrs = []
for i in range(X_feats.shape[1]):
    corr, pval = pointbiserialr(y_ri, X_feats[:, i])
    pb_corrs.append(abs(corr))

# Results table
predictive_power = pd.DataFrame({
    'Feature': ENHANCED_FEATURE_NAMES,
    'MI (intensity)': mi_scores,
    'F-stat (intensity)': f_scores,
    'F p-value': f_pvals,
    '|r_pb| (RI binary)': pb_corrs,
}).sort_values('MI (intensity)', ascending=False)

display(predictive_power.round(4))

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
order = predictive_power['Feature'].values

axes[0].barh(order, predictive_power['MI (intensity)'].values, color='steelblue')
axes[0].set_title('Mutual Information (vs Intensity)', fontweight='bold')

axes[1].barh(order, np.log10(predictive_power['F-stat (intensity)'].values + 1), color='coral')
axes[1].set_title('log₁₀(F-statistic) (vs Intensity)', fontweight='bold')

axes[2].barh(order, predictive_power['|r_pb| (RI binary)'].values, color='seagreen')
axes[2].set_title('|Point-biserial r| (vs RI)', fontweight='bold')

for ax in axes: ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 6. Model Integration & Training
Extend the physics encoder to include new features, then train LOBO.

In [ ]:
# ── Model & Dataset (with extended physics features) ──
ORIG_PHYS_DIM = 8
NEW_PHYS_DIM = ORIG_PHYS_DIM + len(ENHANCED_FEATURE_NAMES)  # 8 + 15 = 23

class TCNDDatasetEnhanced(Dataset):
    """Dataset with enhanced engineered features."""
    def __init__(self, samples, use_3d=True, use_env=True, use_enhanced=True):
        self.samples = samples
        self.use_3d, self.use_env, self.use_enhanced = use_3d, use_env, use_enhanced
        self.basin_to_idx = {b:i for i,b in enumerate(BASINS)}
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        item = {
            'data_1d': torch.tensor(s['data1d'], dtype=torch.float32),
            'y_intensity': torch.tensor(s['y_intensity'], dtype=torch.long),
            'y_direction': torch.tensor(s['y_direction'], dtype=torch.long),
            'basin_idx': torch.tensor(self.basin_to_idx[s['basin']], dtype=torch.long),
        }
        if self.use_3d:
            item['data_3d'] = torch.tensor(load_data3d(s['data3d_path']), dtype=torch.float32)
        else:
            item['data_3d'] = torch.zeros(13, 81, 81)
        if self.use_env:
            env = np.load(s['env_path'], allow_pickle=True).item()
            item['env_data'] = torch.tensor(build_env_vector(env), dtype=torch.float32)
        else:
            item['env_data'] = torch.zeros(77)
        # Original physics features
        sst_anom = s['data1d'][3] * 0.5
        wind_shear = abs(s['data1d'][2] - s['data1d'][3])
        lat_rad = abs(s['data1d'][1]) * 0.3
        coriolis = 2 * 7.2921e-5 * np.sin(np.clip(lat_rad, -np.pi/2, np.pi/2))
        mpi_proxy = max(0, sst_anom * 2 - wind_shear)
        bl_moisture = max(0, 1.0 - abs(s['data1d'][2]))
        outflow_temp = -0.5 * abs(s['data1d'][1])
        steering = s['data1d'][0] * 0.1
        current_int = s['data1d'][3]
        orig_phys = [sst_anom, wind_shear, coriolis, mpi_proxy,
                     bl_moisture, outflow_temp, steering, current_int]
        # Enhanced features
        if self.use_enhanced:
            enhanced = compute_enhanced_features(s)
            enhanced_vals = [enhanced[k] for k in ENHANCED_FEATURE_NAMES]
            all_phys = orig_phys + enhanced_vals
        else:
            all_phys = orig_phys
        item['phys_features'] = torch.tensor(all_phys, dtype=torch.float32)
        return item

print(f"Enhanced physics feature dimension: {NEW_PHYS_DIM} (was {ORIG_PHYS_DIM})")

In [ ]:
# ── Shared model architecture (from base notebook) ──
class ConvBnRelu(nn.Sequential):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__(nn.Conv2d(in_c,out_c,k,stride=s,padding=p,bias=False),
                         nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

class SpatialEncoder(nn.Module):
    def __init__(self, in_ch=13, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            ConvBnRelu(in_ch,32,k=7,s=2,p=3), ConvBnRelu(32,64,k=3,s=2,p=1),
            ConvBnRelu(64,128,k=3,s=2,p=1), ConvBnRelu(128,256,k=3,s=2,p=1),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.proj = nn.Sequential(nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.proj(self.net(x))

class TrackEncoder(nn.Module):
    def __init__(self, in_dim=4, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64,embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.net(x)

class EnvEncoder(nn.Module):
    def __init__(self, in_dim=77, embed_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128,embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.net(x)

class PhysicsEncoder(nn.Module):
    def __init__(self, in_dim=8, phys_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64,phys_dim), nn.LayerNorm(phys_dim))
    def forward(self, x): return self.net(x)

class MultimodalBackbone(nn.Module):
    def __init__(self, spatial_embed=128, track_embed=32, env_embed=64,
                 phys_dim=32, phys_in_dim=8, final_dim=128, use_3d=True, use_env=True):
        super().__init__()
        self.use_3d, self.use_env, self.phys_dim = use_3d, use_env, phys_dim
        self.spatial_enc = SpatialEncoder(embed_dim=spatial_embed) if use_3d else None
        self.track_enc = TrackEncoder(embed_dim=track_embed)
        self.env_enc = EnvEncoder(embed_dim=env_embed) if use_env else None
        self.phys_enc = PhysicsEncoder(in_dim=phys_in_dim, phys_dim=phys_dim)
        fused_in = track_embed + (spatial_embed if use_3d else 0) + (env_embed if use_env else 0)
        self.projector = nn.Sequential(
            nn.Linear(fused_in, final_dim*2), nn.LayerNorm(final_dim*2), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(final_dim*2, final_dim), nn.LayerNorm(final_dim))
        self.env_dim = final_dim - phys_dim
        self.final_dim = final_dim
        self.phys_align = nn.Linear(phys_dim, phys_dim)
    def forward(self, batch):
        parts = []
        if self.use_3d: parts.append(self.spatial_enc(batch['data_3d']))
        parts.append(self.track_enc(batch['data_1d']))
        if self.use_env: parts.append(self.env_enc(batch['env_data']))
        z_full = self.projector(torch.cat(parts, dim=-1))
        z_env = z_full[:, :self.env_dim]; z_phys_raw = z_full[:, self.env_dim:]
        phys_enc_out = self.phys_enc(batch['phys_features'])
        z_phys = z_phys_raw + self.phys_align(phys_enc_out)
        z = torch.cat([z_env, z_phys], dim=-1)
        return {'z':z, 'z_phys':z_phys, 'z_phys_raw':z_phys_raw, 'z_env':z_env}

class TaskHeads(nn.Module):
    def __init__(self, in_dim=128, n_int=4, n_dir=8):
        super().__init__()
        self.int_head = nn.Sequential(nn.Linear(in_dim,64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64,n_int))
        self.dir_head = nn.Sequential(nn.Linear(in_dim,64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64,n_dir))
    def forward(self, z): return self.int_head(z), self.dir_head(z)

class TCModelEnhanced(nn.Module):
    def __init__(self, use_3d=True, use_env=True, phys_dim=32, phys_in_dim=8, final_dim=128):
        super().__init__()
        self.backbone = MultimodalBackbone(use_3d=use_3d, use_env=use_env,
            phys_dim=phys_dim, phys_in_dim=phys_in_dim, final_dim=final_dim)
        self.heads = TaskHeads(in_dim=final_dim)
    def forward(self, batch):
        feat = self.backbone(batch)
        li, ld = self.heads(feat['z'])
        return {**feat, 'logits_intensity':li, 'logits_direction':ld}

print("✓ Enhanced model architecture defined")

In [ ]:
# ── Training utilities ──
def task_loss(li, ld, yi, yd, iw=1.0, dw=0.5):
    return iw * F.cross_entropy(li, yi) + dw * F.cross_entropy(ld, yd)

class ERM:
    def compute_loss(self, batches, model):
        losses = []
        for b in batches.values():
            out = model(b)
            losses.append(task_loss(out['logits_intensity'], out['logits_direction'],
                                   b['y_intensity'], b['y_direction']))
        erm = torch.stack(losses).mean()
        return erm, {'erm_loss': erm.item()}

def make_lobo_split(all_samples, target_basin, val_frac=0.15):
    source = [s for s in all_samples if s['basin'] != target_basin]
    target = [s for s in all_samples if s['basin'] == target_basin]
    np.random.shuffle(source)
    n_val = int(len(source) * val_frac)
    return source[n_val:], source[:n_val], target

def make_per_basin_loaders(samples, batch_size, dataset_cls, **ds_kwargs):
    by_basin = defaultdict(list)
    for s in samples: by_basin[s['basin']].append(s)
    loaders = {}
    for b, samps in by_basin.items():
        ds = dataset_cls(samps, **ds_kwargs)
        safe_bs = min(batch_size, len(ds))
        loaders[b] = DataLoader(ds, batch_size=safe_bs, shuffle=True, drop_last=True, num_workers=0)
    return loaders

def train_epoch(model, method, train_loaders, optimizer, device):
    model.train()
    env_iters = {b: iter(l) for b, l in train_loaders.items()}
    n_batches = max(len(l) for l in train_loaders.values())
    epoch_loss = 0
    for _ in range(n_batches):
        batches = {}
        for b, it in env_iters.items():
            try: batch = next(it)
            except StopIteration:
                env_iters[b] = iter(train_loaders[b]); batch = next(env_iters[b])
            batches[b] = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
        optimizer.zero_grad()
        loss, _ = method.compute_loss(batches, model)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / n_batches

def evaluate(model, loader, device):
    model.eval()
    preds_int, labels_int = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
            out = model(batch)
            preds_int.extend(out['logits_intensity'].argmax(-1).cpu().numpy())
            labels_int.extend(batch['y_intensity'].cpu().numpy())
    pi, li = np.array(preds_int), np.array(labels_int)
    acc = float((pi == li).mean())
    wf1 = float(f1_score(li, pi, average='weighted', zero_division=0))
    mf1 = float(f1_score(li, pi, average='macro', zero_division=0))
    prec, rec, f1, _ = precision_recall_fscore_support(li, pi, labels=[RI_CLASS], zero_division=0)
    return {'acc': acc, 'wf1': wf1, 'macro_f1': mf1, 'ri_f1': float(f1[0])}

def run_fe_experiment(name, all_samples, target_basin, use_enhanced, epochs, use_3d, use_env, bs):
    phys_in = NEW_PHYS_DIM if use_enhanced else ORIG_PHYS_DIM
    phys_dim = 32; final_dim = 128 if use_3d else (96 if use_env else 48)
    train_s, val_s, test_s = make_lobo_split(all_samples, target_basin)
    ds_cls = TCNDDatasetEnhanced
    ds_kwargs = dict(use_3d=use_3d, use_env=use_env, use_enhanced=use_enhanced)
    train_loaders = make_per_basin_loaders(train_s, bs, ds_cls, **ds_kwargs)
    val_loader = DataLoader(ds_cls(val_s, **ds_kwargs), batch_size=bs, num_workers=0)
    test_loader = DataLoader(ds_cls(test_s, **ds_kwargs), batch_size=bs, num_workers=0)
    model = TCModelEnhanced(use_3d=use_3d, use_env=use_env, phys_dim=phys_dim,
                            phys_in_dim=phys_in, final_dim=final_dim).to(DEVICE)
    method = ERM()
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    best_wf1, best_state = -1, None
    for ep in range(1, epochs+1):
        t0 = time.time()
        loss = train_epoch(model, method, train_loaders, optimizer, DEVICE)
        val_r = evaluate(model, val_loader, DEVICE)
        if val_r['wf1'] > best_wf1:
            best_wf1 = val_r['wf1']; best_state = copy.deepcopy(model.state_dict())
        print(f"  [{name}→{target_basin}] Ep {ep}/{epochs} | loss={loss:.4f} | "
              f"acc={val_r['acc']:.3f} | wF1={val_r['wf1']:.3f} | RI-F1={val_r['ri_f1']:.3f} | {time.time()-t0:.1f}s")
    if best_state: model.load_state_dict(best_state)
    test_r = evaluate(model, test_loader, DEVICE)
    print(f"  → Test: acc={test_r['acc']:.3f} | wF1={test_r['wf1']:.3f} | RI-F1={test_r['ri_f1']:.3f}")
    return {'name': name, 'enhanced': use_enhanced, **test_r}

print("✓ Training utilities defined")

## 7. Feature Impact Comparison
Train with and without enhanced features.

In [ ]:
FE_RESULTS = []

print("\n" + "="*60)
print("Experiment: BASELINE (original features only)")
print("="*60)
r = run_fe_experiment('Baseline', ALL_SAMPLES, TARGET_BASIN,
                      use_enhanced=False, epochs=EPOCHS, use_3d=USE_3D, use_env=USE_ENV, bs=BATCH_SIZE)
FE_RESULTS.append(r)

print("\n" + "="*60)
print("Experiment: ENHANCED (with engineered features)")
print("="*60)
r = run_fe_experiment('Enhanced', ALL_SAMPLES, TARGET_BASIN,
                      use_enhanced=True, epochs=EPOCHS, use_3d=USE_3D, use_env=USE_ENV, bs=BATCH_SIZE)
FE_RESULTS.append(r)

## 8. Results & Recommendations

In [ ]:
print("═" * 80)
print("FEATURE ENGINEERING RESULTS")
print("═" * 80)
print(f"{'Experiment':<20} {'Acc':>8} {'wF1':>8} {'Macro F1':>8} {'RI-F1':>8}")
print("-" * 60)
for r in FE_RESULTS:
    print(f"{r['name']:<20} {r['acc']:>8.3f} {r['wf1']:>8.3f} {r['macro_f1']:>8.3f} {r['ri_f1']:>8.3f}")

# Compute deltas
if len(FE_RESULTS) >= 2:
    base, enh = FE_RESULTS[0], FE_RESULTS[1]
    print(f"\nΔ (Enhanced - Baseline):")
    print(f"  Acc:  {enh['acc']-base['acc']:+.3f}")
    print(f"  wF1:  {enh['wf1']-base['wf1']:+.3f}")
    print(f"  RI-F1: {enh['ri_f1']-base['ri_f1']:+.3f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['acc', 'wf1', 'macro_f1', 'ri_f1']
metric_labels = ['Accuracy', 'Weighted F1', 'Macro F1', 'RI F1']
x = np.arange(len(metrics))
width = 0.3

for i, r in enumerate(FE_RESULTS):
    vals = [r[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=r['name'])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.3f}', ha='center', fontsize=8)

ax.set_xticks(x + width/2); ax.set_xticklabels(metric_labels)
ax.set_title(f'Feature Engineering Impact (Target: {TARGET_BASIN})', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

print("\n" + "═" * 80)
print("RECOMMENDATIONS")
print("═" * 80)
print("""
Key engineered features to consider keeping:
• shear_int_interaction — captures the physics of shear-intensity coupling
• intensification_potential — proxy for MPI deficit (room to intensify)
• coriolis_x_wind — latitude-intensity interaction (basin-invariant)
• favourable_shear — binary indicator of low-shear environment
• wind_pres_ratio — deviation from standard wind-pressure relationship

Features to potentially EXCLUDE (basin-specific signals):
• translation_proxy — heavily correlated with longitude (basin-specific)
• sst_bin — discretised proxy may not add value over continuous version

→ Feed these recommendations into the Feature Selection notebook.
""")
print("✅ Feature Engineering analysis complete!")